In [ ]:
"""Clinical utility: observed prevalence by risk zone"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from statsmodels.stats.proportion import proportion_confint

In [ ]:
OOF_PATH, OUT_DIR = "Results_MoE/oof_predictions.csv", "Results_Clinical_Utility"
os.makedirs(OUT_DIR, exist_ok=True)

oof = pd.read_csv(OOF_PATH)
oof["gender"] = oof["gender"].str.lower().str.strip()

In [ ]:
# Risk-zone figure, reads only oof_predictions.csv
ZONE_LOW, ZONE_HIGH, BIN_WIDTH = 40, 65, 10
ZC = {"green": "#2ca02c", "yellow": "#f0c808", "red": "#d62728"}
bins = np.arange(0, 100 + BIN_WIDTH, BIN_WIDTH)
xs = np.linspace(0, 100, 400)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
ppv_rows = []
for col, sex in enumerate(["female", "male"]):
    g = oof[oof.gender == sex]
    y, s, b = g.y_any.to_numpy().astype(int), g.predicted_composite_score.to_numpy(float), g.burden_score.to_numpy()
    top, bot = axes[0, col], axes[1, col]
    for ax in (top, bot):
        ax.axvspan(0, ZONE_LOW, color=ZC["green"], alpha=0.08)
        ax.axvspan(ZONE_LOW, ZONE_HIGH, color=ZC["yellow"], alpha=0.10)
        ax.axvspan(ZONE_HIGH, 100, color=ZC["red"], alpha=0.08)
    for m, color, label in [(y == 0, "#0072B2", "No condition (burden 0)"),
                            (y == 1, "#D55E00", "Any condition (burden \u2265 1)")]:
        kde = gaussian_kde(s[m])
        top.plot(xs, kde(xs), color=color, lw=1.8, label=label)
        top.fill_between(xs, kde(xs), color=color, alpha=0.15)
    top.set_title(f"{sex.capitalize()} (N={len(y)}, prevalence {y.mean():.0%})")
    top.legend(fontsize=8, loc="upper right")

    idx = np.clip(np.digitize(s, bins) - 1, 0, len(bins) - 2)      # score = 100 goes into the last bin
    for k in range(len(bins) - 1):
        m = idx == k
        if m.sum() == 0:
            continue
        p, c = y[m].mean() * 100, (bins[k] + bins[k + 1]) / 2
        lo, hi = proportion_confint(y[m].sum(), m.sum(), method="wilson")
        bot.bar(c, p, width=BIN_WIDTH * 0.85, color="#D55E00", alpha=0.7, edgecolor="white")
        bot.errorbar(c, p, yerr=[[p - 100 * lo], [100 * hi - p]], fmt="none", ecolor="black", elinewidth=1, capsize=3)
        bot.text(c, 100 * hi + 2, f"n={m.sum()}", ha="center", fontsize=6.5, color="#555555")
    bot.plot([0, 100], [0, 100], "k--", lw=0.9, label="Perfect calibration")
    bot.legend(fontsize=8, loc="upper left")
    bot.set_xlabel("Predicted risk score (0-100)")
    bot.set_xlim(0, 100); bot.set_ylim(0, 112)

    for name, mask in [(f"score \u2265 {ZONE_LOW}", s >= ZONE_LOW), (f"score \u2265 {ZONE_HIGH}", s >= ZONE_HIGH)]:
        ppv_lo = proportion_confint(y[mask].sum(), mask.sum(), method="wilson")
        npv_lo = proportion_confint((1 - y[~mask]).sum(), (~mask).sum(), method="wilson")
        ppv_rows.append(dict(sex=sex, cutoff=name, PPV=y[mask].mean(), PPV_lo=ppv_lo[0], PPV_hi=ppv_lo[1],
                             NPV=(1 - y[~mask]).mean(), NPV_lo=npv_lo[0], NPV_hi=npv_lo[1],
                             pct_burden0_below=100 * (b[~mask] == 0).mean()))
axes[0, 0].set_ylabel("Density")
axes[1, 0].set_ylabel("% with burden \u2265 1 (95% Wilson CI)")
fig.suptitle(f"Predicted risk score by illustrative risk zone (green < {ZONE_LOW}, yellow {ZONE_LOW}-{ZONE_HIGH}, "
             f"red \u2265 {ZONE_HIGH})", fontsize=11)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/risk_zones_figure.pdf", bbox_inches="tight")
fig.savefig(f"{OUT_DIR}/risk_zones_figure.png", dpi=300, bbox_inches="tight")
plt.show()

pv = pd.DataFrame(ppv_rows)
pv.round(3).to_csv(f"{OUT_DIR}/zone_cutoffs_ppv_npv.csv", index=False)
print(pv.round(3).to_string(index=False))